# aus-agent-pilot injection support comparison

Compare RAGDoll citation-support judgments for the baseline and all injection variants.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_colwidth', 160)

cwd = Path.cwd().resolve()
repo = next((p for p in [cwd, *cwd.parents] if (p / 'evaluation-results').is_dir()), None)
assert repo is not None, 'Could not locate repository root'
pilot = repo / 'evaluation-results' / 'aus-agent-pilot'

RUNS = {
    'baseline': 'support-bedrock-20',
    'inject': 'support-bedrock-20_inject',
    'inject_rus': 'support-bedrock-20_inject_rus',
    'inject_thai': 'support-bedrock-20_inject_thai',
    'inject_vi': 'support-bedrock-20_inject_vi',
}
LABEL_SCORE = {'FS': 1.0, 'PS': 0.5, 'NS': 0.0}

rows = []
for variant, directory in RUNS.items():
    path = pilot / directory / 'judgments.jsonl'
    assert path.exists(), f'Missing {path}'
    for line_number, line in enumerate(path.open(encoding='utf-8'), 1):
        if not line.strip():
            continue
        item = json.loads(line)
        meta = item.get('metadata') if isinstance(item.get('metadata'), dict) else {}
        rows.append({
            'variant': variant,
            'task_id': item.get('task_id'),
            'status': item.get('status'),
            'label': item.get('support_label'),
            'score': LABEL_SCORE.get(item.get('support_label')),
            'run_id': str(meta.get('run_id', '')),
            'qid': str(meta.get('topic_id', '')),
            'sentence_index': meta.get('sentence_index'),
            'citation_index': meta.get('citation_index'),
            'docid': str(meta.get('docid', '')),
            'statement': item.get('statement', ''),
            'citation': item.get('citation', ''),
            'error': item.get('error'),
            'source_line': line_number,
        })

df = pd.DataFrame(rows)
variant_order = list(RUNS)
df['variant'] = pd.Categorical(df.variant, categories=variant_order, ordered=True)
print(f'Loaded {len(df):,} judgments across {len(RUNS)} variants.')

## Run health and coverage

In [ ]:
health = (df.groupby('variant', observed=False)
          .agg(rows=('task_id', 'size'), completed=('status', lambda s: int((s == 'completed').sum())),
               judge_errors=('score', lambda s: int(s.isna().sum())),
               runs=('run_id', 'nunique'), topics=('qid', 'nunique'), documents=('docid', 'nunique')))
display(health)

errors = df[df.score.isna()][['variant', 'task_id', 'status', 'error']]
if len(errors):
    display(errors.reset_index(drop=True))

## Overall support comparison

In [ ]:
valid = df[df.label.isin(LABEL_SCORE)].copy()
overall = (valid.groupby('variant', observed=False)
           .agg(judgments=('score', 'size'), mean_support=('score', 'mean'),
                full_support_rate=('label', lambda s: (s == 'FS').mean()),
                partial_support_rate=('label', lambda s: (s == 'PS').mean()),
                no_support_rate=('label', lambda s: (s == 'NS').mean())))
overall['partial_or_full_rate'] = overall.full_support_rate + overall.partial_support_rate
display(overall)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
composition = (valid.groupby(['variant', 'label'], observed=False).size().unstack(fill_value=0)
               .reindex(index=variant_order, columns=['FS', 'PS', 'NS'], fill_value=0))
composition.div(composition.sum(axis=1), axis=0).plot.bar(
    stacked=True, ax=axes[0], color=['#2e7d32', '#f9a825', '#c62828'])
axes[0].set(title='Support-label composition', xlabel='', ylabel='Share', ylim=(0, 1))
axes[0].tick_params(axis='x', rotation=25)
overall[['mean_support', 'full_support_rate']].plot.bar(ax=axes[1], color=['#1565c0', '#2e7d32'])
axes[1].set(title='Support quality by variant', xlabel='', ylabel='Rate', ylim=(0, 1))
axes[1].tick_params(axis='x', rotation=25)
plt.tight_layout(); plt.show()

## Pair alignment

Injection can shift sentence indices. Citations are therefore paired using `run_id + qid + docid + occurrence`, where occurrence is the citation's order among repeated uses of the same document.

In [ ]:
valid = valid.sort_values(['variant', 'run_id', 'qid', 'sentence_index', 'citation_index', 'source_line'])
valid['occurrence'] = valid.groupby(['variant', 'run_id', 'qid', 'docid'], observed=False).cumcount()
key = ['run_id', 'qid', 'docid', 'occurrence']
base = valid[valid.variant == 'baseline'][key + ['label', 'score', 'statement', 'citation']].rename(columns={
    'label': 'baseline_label', 'score': 'baseline_score', 'statement': 'baseline_statement',
    'citation': 'baseline_citation'})

paired_parts = []
alignment_rows = []
for variant in variant_order[1:]:
    other = valid[valid.variant == variant][key + ['label', 'score', 'statement']].rename(columns={
        'label': 'injected_label', 'score': 'injected_score', 'statement': 'injected_statement'})
    joined = base.merge(other, on=key, how='outer', indicator=True)
    joined['variant'] = variant
    joined['delta'] = joined.injected_score - joined.baseline_score
    paired_parts.append(joined)
    counts = joined._merge.value_counts()
    alignment_rows.append({
        'variant': variant,
        'paired': int(counts.get('both', 0)),
        'baseline_only': int(counts.get('left_only', 0)),
        'injected_only': int(counts.get('right_only', 0)),
    })
paired = pd.concat(paired_parts, ignore_index=True)
display(pd.DataFrame(alignment_rows).set_index('variant'))

## Paired score changes

In [ ]:
both = paired[paired._merge == 'both'].copy()
paired_summary = (both.groupby('variant')
                  .agg(pairs=('delta', 'size'), mean_delta=('delta', 'mean'),
                       improved=('delta', lambda s: int((s > 0).sum())),
                       unchanged=('delta', lambda s: int((s == 0).sum())),
                       degraded=('delta', lambda s: int((s < 0).sum()))))
paired_summary['degradation_rate'] = paired_summary.degraded / paired_summary.pairs
display(paired_summary)

ax = paired_summary[['improved', 'unchanged', 'degraded']].plot.bar(
    stacked=True, figsize=(10, 4), color=['#2e7d32', '#90a4ae', '#c62828'])
ax.set(title='Paired citation verdict changes from baseline', xlabel='', ylabel='Citation pairs')
ax.tick_params(axis='x', rotation=25)
plt.tight_layout(); plt.show()

## Verdict transition matrices

In [ ]:
LABELS = ['FS', 'PS', 'NS']
for variant in variant_order[1:]:
    subset = both[both.variant == variant]
    matrix = pd.crosstab(subset.baseline_label, subset.injected_label).reindex(
        index=LABELS, columns=LABELS, fill_value=0)
    display(pd.io.formats.style.Styler(matrix).set_caption(f'Baseline → {variant}'))

## Per-topic effects

In [ ]:
topic_delta = (both.groupby(['variant', 'run_id', 'qid'], as_index=False)
               .agg(pairs=('delta', 'size'), baseline_mean=('baseline_score', 'mean'),
                    injected_mean=('injected_score', 'mean'), mean_delta=('delta', 'mean'),
                    degraded=('delta', lambda s: int((s < 0).sum()))))
display(topic_delta.sort_values(['mean_delta', 'degraded']).reset_index(drop=True))

pivot = topic_delta.pivot_table(index=['run_id', 'qid'], columns='variant', values='mean_delta')
display(pivot.style.background_gradient(cmap='RdYlGn', vmin=-1, vmax=1).set_caption('Mean support-score delta by topic'))

## Changed-verdict review

Adjust `VARIANT` and `CHANGE` to inspect the underlying statements.

In [ ]:
VARIANT = 'inject_vi'
CHANGE = 'degraded'  # 'degraded', 'improved', or 'changed'

review = both[both.variant == VARIANT].copy()
if CHANGE == 'degraded': review = review[review.delta < 0]
elif CHANGE == 'improved': review = review[review.delta > 0]
elif CHANGE == 'changed': review = review[review.delta != 0]
else: raise ValueError("CHANGE must be 'degraded', 'improved', or 'changed'")
display(review[[
    'run_id', 'qid', 'docid', 'occurrence', 'baseline_label', 'injected_label', 'delta',
    'baseline_statement', 'injected_statement', 'baseline_citation'
]].sort_values(['delta', 'run_id', 'qid']).reset_index(drop=True))